Import bibliotek:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.preprocessing import LabelEncoder
sns.set_theme()

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
#x

Wczytanie pliku:

In [ ]:
df = pd.read_csv('star_classification.csv')
df.head(10)

## Czyszczenie danych

Usunięcie kolumn zawierjących dane porządkowe:

In [ ]:
df.columns

In [ ]:
df = df.drop(columns=['obj_ID','run_ID','rerun_ID','cam_col','field_ID','spec_obj_ID','plate','MJD','fiber_ID'])

In [ ]:
df

Sprawdzenie, czy występują wiersze bez wartości:

In [ ]:
df.info()

In [ ]:
df.dropna(inplace= True)

Sprawdzenie czy są duplikaty

In [ ]:
print(df.duplicated().sum())

Sprawdzenie ilości obiektów w bazie:

In [ ]:
df['class'].value_counts()

In [ ]:
sns.histplot(df['class'], bins=20)

Zmniejszenie ilości typu - GALAXY:

In [ ]:
kolumny = [ 'u', 'g', 'r', 'i', 'z', 'redshift']
sns.boxplot(data=df[df['class'] == 'GALAXY'][kolumny])

Usunięcie kilku rekordów z najpowszechniejszymi danymi:

In [ ]:
galaxies = df[df['class'] == 'GALAXY']

In [ ]:
galaxies

In [ ]:

mean_val = galaxies[kolumny].mean()
galaxies['distance_from_mean'] = abs(galaxies[kolumny] - mean_val).sum(axis=1)
galaxies_to_keep = galaxies.sort_values('distance_from_mean').iloc[16000:]

lub losowe usuniecie

In [ ]:
#galaxies_to_keep = galaxies.sample(n=45000, random_state=69)

In [ ]:
galaxies_to_keep

Wymiana na nowy zbiór galaktyk:

In [ ]:
do_usuniecia = df[df['class'] == 'GALAXY'].index
df = df.drop(do_usuniecia)

In [ ]:
df2 = pd.concat([galaxies_to_keep, df])

In [ ]:
df2 = df2.drop('distance_from_mean', axis=1)

In [ ]:
sns.boxplot(data=df2[df2['class'] == 'GALAXY'][kolumny])

In [ ]:
sns.histplot(df2['class'], bins=20)

Sprawdzenie poprawności rekordów typu STAR

In [ ]:
sns.boxplot(data=df2[df2['class'] == 'STAR'][kolumny])

"Ukryte" braki danych

In [ ]:
df2.replace(-9999, np.nan, inplace=True)
df2.dropna(inplace=True)


In [ ]:
sns.boxplot(data=df2[df2['class'] == 'STAR'][kolumny])

Sprawdzenie wartości obiektów QSO

In [ ]:
sns.boxplot(data=df2[df2['class'] == 'QSO'][kolumny])

One-Hot-Encoding

In [ ]:
#df_encoded = pd.get_dummies(df2, columns=['class'])

lub label encoding

In [ ]:
le = LabelEncoder()
df_encoded = df2
df_encoded['target'] = le.fit_transform(df2['class']) 
df_encoded =df_encoded.drop('class',axis = 1)

In [ ]:
df_encoded.tail(8)

Ostateczne sprawdzenie, czy zbiór jest gotowy:

In [ ]:
df_encoded.isnull().sum()

In [ ]:
df_encoded.info()

Sprawdzenie, czy kolumny nie mają za dużej korelacji:

In [ ]:
corr = df_encoded.corr()
plt.figure(figsize=(10, 8)) 
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

In [ ]:
df_encoded.describe()

## Budowa modelu

In [ ]:
X = df_encoded[['u','g','r','i','z','redshift','alpha','delta']]
Y = df_encoded[['target']]

In [ ]:
X_train_full, X_test, Y_train_full, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_full, Y_train_full, test_size= 0.25, random_state= 42
                                                  )

train_full - zbiór treningowy + zbiór walidacyjny

train - zbiór treningowy

val - zbiór walidacyjny

test - zbiór testowy

## Stworzenie i trenowanie modelu

In [ ]:
xgb_model = xgb.XGBClassifier()

In [ ]:
param = {
    'n_estimators': [50, 100, 150, 200, 250, 300],       # ile drzewek zbudować
    'max_depth': [2, 3],                                 # jak głębokie mogą być drzewka
    'learning_rate': [0.1, 0.2, 0.5],                    # jak szybko się uczy 
    'random_state': [42]
}

In [ ]:
model = GridSearchCV(estimator= xgb_model, param_grid=param, cv=5, n_jobs=-1, verbose=2)

cv - walidacja krzyzowa 

n_jobs - ile rdzeni procesora do obliczen

verbose - jak bardzo sczegolowe info 

In [ ]:
model.fit(X_train, Y_train)

In [ ]:
best_model =model.best_estimator_

### Najlepsze parametry

In [ ]:
print(f"Najlepsze parametry: {model.best_params_}")
print(f"Najlepszy wynik (accuracy): {model.best_score_}")

# Zbiór treningowy i walidacyjny

## Ocena modelu

In [ ]:
print("Train accuracy:", accuracy_score(Y_train, best_model.predict(X_train)))
print(classification_report(Y_train, best_model.predict(X_train)))

In [ ]:
print("Validation accuracy:", accuracy_score(Y_val, best_model.predict(X_val)))
print(classification_report(Y_val, best_model.predict(X_val)))

# Zbiór testowy

In [ ]:
print("Test accuracy: ", accuracy_score(Y_test, best_model.predict(X_test)))
print(classification_report(Y_test, best_model.predict(X_test)))

## Ważność cech

In [ ]:
xgb.plot_importance(best_model)

plt.show()

## Macierz pomyłek

In [ ]:
cm =  confusion_matrix(Y_test, best_model.predict(X_test))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(cmap="Blues")
plt.show()

## Sprawdzenie przeuczenia

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_test, Y_test, cv=5, scoring='accuracy', n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.plot(train_sizes, train_mean, label='Trening')
plt.plot(train_sizes, test_mean, label='Test')
plt.xlabel('Wielkość zbioru treningowego')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Krzywe uczenia')
plt.show()

## Zapis modelu

In [ ]:
best_model.save_model("model.json")